# Temporal-Difference (TD) learning: the central idea of reinforcement learning

_Reinforcement Learning_

**Don't wait for the episode to end and don't need a model — nudge each state's value toward the very next step's estimate.**

TD (Temporal-Difference) learning is the single most important idea in reinforcement learning.
       It splices together the two older families:

       
         
- From Monte Carlo (see aix-monte-carlo) it takes model-free sampling:
         learn straight from experienced transitions, with no knowledge of the world's dynamics.
         
- From dynamic programming (see ai-value-iteration) it takes bootstrapping:
         update a value using another current value estimate, rather than waiting for a final outcome.
       

---

This notebook is a practice scaffold for the **Temporal-Difference (TD) learning: the central idea of reinforcement learning** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — numpy + gymnasium

In [ ]:
# Colab setup cell:  !pip install gymnasium
import numpy as np
import gymnasium as gym

# ============================================================
# TD(0) PREDICTION of V^pi for a FIXED policy (model-free, online).
#   Update each step:  V(S_t) <- V(S_t) + alpha * [ R_{t+1} + gamma*V(S_{t+1}) - V(S_t) ]
# ============================================================
env = gym.make("FrozenLake-v1", is_slippery=True)   # 4x4 grid, 16 states, 4 actions
nS = env.observation_space.n                          # number of states (16)

# A FIXED policy to evaluate: always take action 2 ("move right").
# (Slippery ice makes the outcome stochastic even with a fixed action.)
RIGHT = 2
def policy(state):
    return RIGHT

# Hyperparameters
alpha   = 0.1     # learning rate (step size toward the TD target)
gamma   = 0.99    # discount factor
episodes = 20000

V = np.zeros(nS)  # value estimate, one number per state; starts at 0

for ep in range(episodes):
    state, _ = env.reset()
    done = False
    while not done:
        action = policy(state)                        # fixed-policy action
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # --- THE TD(0) UPDATE ---
        # If the episode ended, the next state's value contributes 0 (terminal).
        v_next = 0.0 if done else V[next_state]
        td_target = reward + gamma * v_next           # R_{t+1} + gamma*V(S_{t+1})
        td_error  = td_target - V[state]              # delta_t (the surprise)
        V[state] += alpha * td_error                  # nudge toward the target

        state = next_state

env.close()

# Print the learned value of each state as a 4x4 grid.
np.set_printoptions(precision=3, suppress=True)
print("Estimated V^pi for the 'always right' policy (FrozenLake 4x4):")
print(V.reshape(4, 4))
# States near the goal (bottom-right) end up with the highest values;
# holes and far cells stay near 0.

## Visualize the data & results

_Does TD(0) really estimate a state's value with lower variance and faster than Monte Carlo? Compare their error vs. number of episodes on the same state of a tiny random-walk Markov Decision Process (MDP)._

In [ ]:
import numpy as np

# A tiny 5-state random walk (Sutton & Barto). States 0..4 (left..right).
# Off the LEFT end -> terminal, reward 0.  Off the RIGHT end -> terminal, reward +1.
# Each step goes left/right with prob 0.5.  gamma = 1.  Start in the center (state 2).
# Under this equiprobable policy the TRUE values are (i+1)/6, so center (state 2) = 3/6 = 0.5.
N        = 5
TRUE     = np.array([1, 2, 3, 4, 5]) / 6.0   # true V for states 0..4
GAMMA    = 1.0
ALPHA    = 0.1                                # SAME step size for both methods
WATCH    = 2                                  # the center state we track

def run_episode():
    s, traj = 2, []
    while True:
        a  = np.random.randint(0, 2)          # 0 = left, 1 = right
        sp = s + (1 if a == 1 else -1)
        if sp < 0:  traj.append((s, 0.0)); return traj    # left terminal, R=0
        if sp > 4:  traj.append((s, 1.0)); return traj    # right terminal, R=1
        traj.append((s, 0.0)); s = sp

def experiment(method, n_episodes, n_runs=200):
    errs = np.zeros(n_episodes)
    for _ in range(n_runs):
        V = np.full(N, 0.5)                   # optimistic init at 0.5
        for ep in range(n_episodes):
            traj = run_episode()
            if method == "TD":
                for t, (s, r) in enumerate(traj):
                    v_next = V[traj[t+1][0]] if t+1 < len(traj) else 0.0
                    V[s] += ALPHA * (r + GAMMA * v_next - V[s])   # TD(0) update
            else:  # Monte Carlo: target = full return from each visited state
                rewards = [r for (_, r) in traj]
                states  = [s for (s, _) in traj]
                returns = np.cumsum(rewards[::-1])[::-1]          # suffix sums (gamma=1)
                for s, g in zip(states, returns):
                    V[s] += ALPHA * (g - V[s])                    # MC update
            errs[ep] += abs(V[WATCH] - TRUE[WATCH])
    return errs / n_runs

np.random.seed(0)
NEP = 100
td_err = experiment("TD", NEP)
mc_err = experiment("MC", NEP)

# Subsample to <=60 plotted points for the chart.
idx = list(range(0, NEP, 2))
print("true center value:", TRUE[WATCH])
print("TD(0) error  :", [round(td_err[i], 4) for i in idx])
print("MonteCarlo   :", [round(mc_err[i], 4) for i in idx])

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A state has $V(S_t) = 2.0$. The agent steps, gets $R_{t+1} = 1$, lands in $S_{t+1}$ with
            $V(S_{t+1}) = 3.0$. Using $\alpha = 0.5$ and $\gamma = 1.0$, compute the TD target, the TD error
            $\delta_t$, and the new $V(S_t)$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the TD target $R_{t+1} + \gamma V(S_{t+1})$. — _It is the one-step estimate we are pulling toward: $1 + 1.0\times 3.0 = 4.0$._
- Compute the TD error $\delta_t = \text{target} - V(S_t) = 4.0 - 2.0 = 2.0$. — _The surprise: the step turned out worth more than the old estimate._
- Apply the update $V(S_t) \leftarrow V(S_t) + \alpha\,\delta_t = 2.0 + 0.5\times 2.0$. — _Move halfway ($\alpha = 0.5$) toward the target._

**Answer:** Target $= 4.0$, $\delta_t = 2.0$, new $V(S_t) = 2.0 + 0.5\times 2.0 = 3.0$.

</details>

**Problem 2.** Your TD estimate of a state has stopped drifting on average, yet the TD error $\delta_t$ is nonzero
            and flips sign from step to step. Is something broken? Why or why not?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what $\delta_t$ measures: the surprise on one sampled step, not a verdict on the value. — _$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ uses single random samples, so it is noisy._
- Recall the convergence condition: at the true value, $\mathbb{E}[\delta_t] = 0$, not $\delta_t = 0$. — _Only the expected error vanishes; individual errors stay noisy and average to zero._

**Answer:** Nothing is broken. A noisy, sign-flipping $\delta_t$ whose average is near zero is exactly what
                 convergence looks like — the value is right on average. Judge by the mean error over many steps,
                 not one.

</details>